# Amazon Bedrock

A practical refresher on **Amazon Bedrock** — AWS's fully managed service for building generative-AI applications on top of foundation models (FMs) from Anthropic, Meta, Mistral, Cohere, AI21, Stability AI, and Amazon (Titan / Nova), all behind a single API.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Amazon Bedrock is a **serverless, fully managed** API for foundation models. You never provision GPUs, manage model weights, or run inference servers — you call an HTTPS endpoint, AWS runs the model, and you pay per token (on-demand) or per model-unit-hour (provisioned throughput).

### What is it?

Bedrock exposes many FMs through **one consistent SDK** (`boto3`'s `bedrock-runtime` client). The unified **Converse API** gives a single request/response shape across Anthropic Claude, Meta Llama, Mistral, Amazon Nova, and others, so you can swap models by changing one `modelId` string. On top of raw inference, Bedrock bundles managed building blocks: **Knowledge Bases** (RAG), **Agents** (tool-using workflows), **Guardrails** (safety/PII filtering), **fine-tuning**, and **model evaluation**.

### Why use it?

- **No infrastructure to run** — serverless inference; scale to zero, no GPU fleet to babysit.
- **Model choice without lock-in** — one API across many providers; A/B test Claude vs. Llama vs. Nova by changing a string.
- **Data stays in your AWS account** — your prompts/completions are not used to train base models, and traffic can stay on the AWS network via VPC (PrivateLink) endpoints.
- **Enterprise plumbing included** — IAM auth, CloudWatch metrics/logs, CloudTrail audit, KMS encryption, and AWS billing in one place.

### When to use it?

- You are already on AWS and want IAM/VPC/CloudWatch integration rather than a separate vendor account and API key.
- You want managed RAG (Knowledge Bases) or agents/guardrails without wiring them yourself.
- You need data-residency / compliance guarantees inside your AWS account.
- You want to keep model choice open and avoid hard-coding a single provider's SDK.

**When *not* to:** if you need a model not offered on Bedrock, want the absolute newest provider feature on day one (providers' own APIs often ship first), or you self-host for cost/latency reasons (then look at SageMaker or EKS + vLLM).

## Key Features

### Core Capabilities of Amazon Bedrock

| Feature | Description | Benefit |
|---------|-------------|----------|
| Unified Converse API | One request/response schema across all chat models | Swap models by changing `modelId`; no per-provider payload code |
| Streaming (`ConverseStream` / `InvokeModelWithResponseStream`) | Token-by-token server-sent responses | Low time-to-first-token for chat UIs |
| Knowledge Bases | Managed RAG: ingest S3 docs → vector store → `RetrieveAndGenerate` | Production RAG without running an embedding/retrieval stack |
| Agents | Models that call tools (Lambda / OpenAPI actions) in a managed loop | Build tool-using assistants without orchestrating the loop yourself |
| Guardrails | Configurable content, topic, PII, and grounding filters | Consistent safety/compliance policy across every model |
| Custom models | Fine-tuning and continued pre-training on your data | Domain adaptation while keeping the managed serving path |
| Provisioned Throughput | Reserved model units with guaranteed capacity | Predictable latency/cost for steady high-volume traffic |
| Cross-region inference | Inference profiles route across regions for headroom | Higher effective quotas and resilience to regional throttling |

## Architecture Overview

Bedrock has two API surfaces: a **control plane** (`bedrock`) for management — listing models, creating Knowledge Bases, fine-tuning jobs, guardrails — and a **data plane** (`bedrock-runtime`) for inference (`Converse`, `InvokeModel`, and their streaming variants). RAG retrieval/generation lives in a third client, `bedrock-agent-runtime`.

```
        Your app (boto3 / SDK)
                 |  SigV4-signed HTTPS (IAM)
                 v
   +-------------------------------+
   |        Amazon Bedrock         |
   |                               |
   |  bedrock           (control)  |  ListFoundationModels, fine-tune, guardrails
   |  bedrock-runtime   (data)     |  Converse / InvokeModel (+Stream)
   |  bedrock-agent-runtime        |  RetrieveAndGenerate, InvokeAgent
   +---------------+---------------+
                   |
      +------------+-------------------------------+
      |            |               |               |
      v            v               v               v
  Foundation   Guardrails     Knowledge Bases   Agents
    Models                    (S3 -> vector DB) (Lambda tools)
  (Claude, Llama,            OpenSearch /       Action groups
   Nova, Mistral...)         Aurora pgvector
```

### Components

1. **Foundation models**: provider-hosted weights you reference by `modelId` (e.g. `anthropic.claude-3-5-sonnet-20241022-v2:0`) or by an **inference profile** ID (e.g. `us.anthropic.claude-3-5-sonnet-20241022-v2:0`) for cross-region routing.
2. **Knowledge Bases**: a managed ingestion pipeline (S3 → chunk → embed → vector store such as OpenSearch Serverless or Aurora pgvector) plus the `RetrieveAndGenerate` API.
3. **Agents**: an orchestration loop where the model plans, calls **action groups** (Lambda functions or OpenAPI schemas), and synthesizes a final answer.
4. **Guardrails**: a policy object (denied topics, content filters, PII redaction, contextual grounding) you attach to any inference call by `guardrailIdentifier`.

## Installation

### Prerequisites

- An AWS account with **model access granted** in the target region. Foundation models are *off by default* — enable them once in the Bedrock console under **Model access** (Anthropic models require a short use-case form).
- AWS credentials available to boto3 (env vars, `~/.aws/credentials`, SSO, or an instance/task role).
- An IAM principal allowed to call `bedrock:InvokeModel` (and `bedrock:Converse`).
- Python 3.8+ and a recent `boto3` (the Converse API needs `boto3 >= 1.34.x`).

### Installation Steps

In [ ]:
# Install / upgrade the AWS SDK (boto3) and the lower-level botocore.
%pip install -q --upgrade boto3 botocore

## Basic Usage

### Quick Start Example

The recommended entry point is the **Converse API** — one schema for all chat models. The cell below builds the client and a request, and only performs the live call when AWS credentials are present, so the notebook runs cleanly either way.

In [ ]:
import os
import json
import boto3
from botocore.exceptions import BotoCoreError, ClientError

REGION = os.getenv("AWS_REGION", "us-east-1")
# Inference-profile IDs (the 'us.' prefix) enable cross-region routing.
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-5-sonnet-20241022-v2:0")


def has_aws_creds() -> bool:
    try:
        return boto3.Session().get_credentials() is not None
    except Exception:
        return False


def converse(prompt: str, model_id: str = MODEL_ID, region: str = REGION) -> str:
    client = boto3.client("bedrock-runtime", region_name=region)
    resp = client.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        system=[{"text": "You are a concise assistant."}],
        inferenceConfig={"maxTokens": 256, "temperature": 0.2, "topP": 0.9},
    )
    # Usage (tokens) and stopReason come back on every response.
    usage = resp["usage"]
    print(f"tokens in/out: {usage['inputTokens']}/{usage['outputTokens']}")
    return resp["output"]["message"]["content"][0]["text"]


if has_aws_creds():
    try:
        print(converse("In one sentence, what is Amazon Bedrock?"))
    except (BotoCoreError, ClientError) as e:
        print(f"Bedrock call failed (check region/model access): {e}")
else:
    print("No AWS credentials found - skipping live call. Request shape shown above.")

### Discovering available models

Use the **control-plane** client to see which models exist in a region and which you have access to.

In [ ]:
if has_aws_creds():
    try:
        bedrock = boto3.client("bedrock", region_name=REGION)
        models = bedrock.list_foundation_models(byOutputModality="TEXT")["modelSummaries"]
        for m in models[:10]:
            print(f"{m['modelId']:55s} {m.get('providerName', '')}")
    except (BotoCoreError, ClientError) as e:
        print(f"list_foundation_models failed: {e}")
else:
    print("Example modelIds you would see:")
    for mid in [
        "anthropic.claude-3-5-sonnet-20241022-v2:0",
        "anthropic.claude-3-5-haiku-20241022-v1:0",
        "amazon.nova-pro-v1:0",
        "meta.llama3-1-70b-instruct-v1:0",
        "mistral.mistral-large-2407-v1:0",
    ]:
        print(" ", mid)

## Advanced Features

### Streaming, tool use, RAG, and guardrails

#### Streaming with `ConverseStream`

Stream tokens as they are generated for responsive chat UIs. The response is an event stream you iterate over.

In [ ]:
def converse_stream(prompt: str, model_id: str = MODEL_ID, region: str = REGION) -> None:
    client = boto3.client("bedrock-runtime", region_name=region)
    resp = client.converse_stream(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 256},
    )
    for event in resp["stream"]:
        if "contentBlockDelta" in event:
            print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
        elif "metadata" in event:
            print("\n--", event["metadata"]["usage"])  # token usage at the end


if has_aws_creds():
    try:
        converse_stream("List three benefits of serverless inference.")
    except (BotoCoreError, ClientError) as e:
        print(f"stream failed: {e}")
else:
    print("Skipping live stream - iterate resp['stream'] for contentBlockDelta events.")

#### Tool use (function calling)

The Converse API exposes a provider-agnostic `toolConfig`. The model returns a `toolUse` block; you run the tool and feed a `toolResult` back in the next turn.

In [ ]:
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_weather",
                "description": "Get the current weather for a city.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"],
                    }
                },
            }
        }
    ]
}


def run_tool(name: str, args: dict) -> str:
    if name == "get_weather":
        return json.dumps({"city": args["city"], "tempC": 21, "sky": "clear"})
    raise ValueError(f"unknown tool {name}")


if has_aws_creds():
    try:
        client = boto3.client("bedrock-runtime", region_name=REGION)
        messages = [{"role": "user", "content": [{"text": "What's the weather in Paris?"}]}]
        resp = client.converse(modelId=MODEL_ID, messages=messages, toolConfig=tool_config)
        out = resp["output"]["message"]
        messages.append(out)
        if resp["stopReason"] == "tool_use":
            for block in out["content"]:
                if "toolUse" in block:
                    tu = block["toolUse"]
                    result = run_tool(tu["name"], tu["input"])
                    messages.append({
                        "role": "user",
                        "content": [{"toolResult": {
                            "toolUseId": tu["toolUseId"],
                            "content": [{"text": result}],
                        }}],
                    })
            final = client.converse(modelId=MODEL_ID, messages=messages, toolConfig=tool_config)
            print(final["output"]["message"]["content"][0]["text"])
    except (BotoCoreError, ClientError) as e:
        print(f"tool-use call failed: {e}")
else:
    print("Skipping live tool-use demo - flow: converse -> toolUse -> run -> toolResult -> converse.")

#### Knowledge Bases (managed RAG)

Once you have created a Knowledge Base in the console (S3 data source + vector store), one call does retrieval **and** generation:

```python
agent_rt = boto3.client("bedrock-agent-runtime", region_name=REGION)
resp = agent_rt.retrieve_and_generate(
    input={"text": "What is our refund policy?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": "KB123ABC",
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/{MODEL_ID}",
        },
    },
)
print(resp["output"]["text"])
for c in resp["citations"]:
    print(c["retrievedReferences"])  # source chunks for attribution
```

#### Guardrails

Attach a guardrail to any inference call to enforce a content / PII / topic policy:

```python
resp = client.converse(
    modelId=MODEL_ID,
    messages=messages,
    guardrailConfig={"guardrailIdentifier": "gr-abc123", "guardrailVersion": "1"},
)
```

## Use Cases

### Real-world Applications of Amazon Bedrock

#### Use Case 1: Enterprise RAG assistant over internal docs

- **Context**: Support and policy documents sit in S3; staff need grounded, cited answers.
- **Implementation**: Knowledge Base (S3 → OpenSearch Serverless) + `RetrieveAndGenerate`; Guardrails to block PII leakage and off-topic queries.
- **Results**: Production RAG with citations and no self-managed vector / embedding stack.

#### Use Case 2: Tool-using agent for operations

- **Context**: An internal assistant must query inventory and create tickets.
- **Implementation**: Bedrock Agent with action groups backed by Lambda functions (OpenAPI schema); the managed loop plans, calls tools, and replies.
- **Results**: Multi-step automation without hand-rolling the plan / act / observe loop.

#### Use Case 3: High-volume document summarization

- **Context**: Millions of records summarized nightly with predictable latency / cost.
- **Implementation**: Bedrock **Batch inference** (S3-in / S3-out) for the bulk job, or Provisioned Throughput for steady real-time load; a cheap fast model (Claude Haiku / Nova Lite) for the summaries.
- **Results**: Lower cost per token than on-demand at scale, with capacity guarantees.

## Best Practices

### Recommended Practices for Amazon Bedrock

1. **Use the Converse API, not raw `InvokeModel`** — it normalizes payloads across providers, so changing model is a one-line change instead of rewriting request/response parsing.
2. **Reference models by inference profile** (e.g. `us.anthropic...`) — cross-region routing raises effective quotas and survives single-region throttling.
3. **Grant least-privilege IAM** — scope `bedrock:InvokeModel` to specific model ARNs and use VPC interface endpoints (PrivateLink) so traffic never leaves the AWS network.
4. **Right-size the model per task** — route cheap/fast tasks to Haiku/Nova Lite and reserve frontier models (Sonnet / Nova Pro) for hard ones; this is the biggest cost lever.
5. **Enable model invocation logging + Guardrails from day one** — you want CloudWatch/S3 records and a consistent safety policy before, not after, an incident.

## Common Pitfalls

### What to Avoid When Using Amazon Bedrock

1. **`AccessDeniedException` on a model that 'should' work** — model access is per-account, per-region, and off by default. Enable it under **Model access**, and remember it doesn't propagate across regions.
2. **`ThrottlingException` under burst load** — on-demand quotas are modest. Add exponential-backoff retries (botocore's `adaptive` retry mode), use cross-region inference profiles, or move to Provisioned Throughput.
3. **Hard-coding a bare `modelId` in one region** — a region without that model returns `ValidationException`. Prefer inference-profile IDs and parameterize the region.
4. **Forgetting context-window and `maxTokens` limits** — oversized prompts get truncated or rejected; set `maxTokens` deliberately and budget the input.
5. **Assuming all models support every feature** — tool use, vision, and system prompts vary by model. Check the model's documented Converse support before relying on it.

## Performance Optimization

### Optimizing Amazon Bedrock for Production

#### Configuration Tuning

Key levers to optimize:

- **Model tier**: smaller/faster models (Haiku, Nova Lite/Micro) cut latency and cost dramatically — route by task difficulty.
- **Provisioned Throughput**: reserve model units for guaranteed, low-variance latency on steady traffic (and to bypass on-demand throttling).
- **Prompt caching**: cache stable prefixes (long system prompts, retrieved context) so repeated tokens are billed and processed at a steep discount and lower latency.
- **Latency-optimized inference**: some models offer a `performanceConfig={'latency': 'optimized'}` mode for faster responses.
- **`maxTokens` and streaming**: cap output length and stream tokens to minimize perceived latency.

The cell below sets up botocore's adaptive retry mode, which is the single most impactful change for throughput reliability.

In [ ]:
from botocore.config import Config

# Adaptive retries back off and respect throttling; raise max_attempts for bursty load.
cfg = Config(
    retries={"max_attempts": 8, "mode": "adaptive"},
    read_timeout=120,  # generation can take a while for long outputs
    connect_timeout=10,
)

tuned_client = boto3.client("bedrock-runtime", region_name=REGION, config=cfg)
print("Configured bedrock-runtime client with adaptive retries:", cfg.retries)

## Production Deployment

### Deploying Amazon Bedrock in Production

Bedrock itself is serverless — you don't deploy the model. What you deploy is the **client app** plus the **IAM, networking, and IaC** around it.

#### Least-privilege IAM policy

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": [
      "bedrock:InvokeModel",
      "bedrock:InvokeModelWithResponseStream",
      "bedrock:Converse",
      "bedrock:ConverseStream"
    ],
    "Resource": [
      "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-5-sonnet-20241022-v2:0",
      "arn:aws:bedrock:*:123456789012:inference-profile/us.anthropic.claude-3-5-sonnet-20241022-v2:0"
    ]
  }]
}
```

#### Private connectivity (VPC interface endpoint)

```bash
# Keep Bedrock traffic on the AWS network via PrivateLink.
aws ec2 create-vpc-endpoint \
  --vpc-id vpc-0abc123 \
  --service-name com.amazonaws.us-east-1.bedrock-runtime \
  --vpc-endpoint-type Interface \
  --subnet-ids subnet-0aaa subnet-0bbb \
  --security-group-ids sg-0ccc
```

#### Containerized client (Docker)

```dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir boto3 fastapi uvicorn
COPY app.py .
# No API keys baked in: use the task/instance IAM role for credentials.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]
```

#### Kubernetes (EKS) deployment with IAM Roles for Service Accounts

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: bedrock-app
spec:
  replicas: 3
  selector:
    matchLabels: { app: bedrock-app }
  template:
    metadata:
      labels: { app: bedrock-app }
    spec:
      serviceAccountName: bedrock-sa   # IRSA-bound role with bedrock:InvokeModel
      containers:
        - name: app
          image: 123456789012.dkr.ecr.us-east-1.amazonaws.com/bedrock-app:latest
          env:
            - name: AWS_REGION
              value: us-east-1
          ports:
            - containerPort: 8080
```

## Monitoring and Observability

### Monitoring Amazon Bedrock in Production

#### Key Metrics to Track

Bedrock publishes metrics to the **`AWS/Bedrock`** CloudWatch namespace:

- **`Invocations`** and **`InvocationClientErrors` / `InvocationServerErrors`**: call volume and error rates (watch for throttling).
- **`InvocationLatency`**: end-to-end latency; alarm on p99 regressions.
- **`InputTokenCount` / `OutputTokenCount`**: token usage — your direct cost driver; aggregate for spend forecasting.
- **`InvocationThrottles`**: a leading indicator that you need higher quotas or provisioned throughput.

#### Logging Best Practices

- **Enable model invocation logging** (console → Settings, or `PutModelInvocationLoggingConfiguration`) to capture full request/response (and optionally embeddings) to S3 and/or CloudWatch Logs.
- **Use CloudTrail** for the control plane (who created/changed guardrails, KBs, fine-tuning jobs) for audit.
- **Log structured request IDs** (`response['ResponseMetadata']['RequestId']`) so app logs correlate with AWS-side records.
- **Attach Guardrails trace** (`trace='ENABLED'`) when debugging to see exactly which policy blocked a request.

## Troubleshooting

### Common Issues with Amazon Bedrock

#### Issue 1: `AccessDeniedException` when invoking a model

**Symptoms**: `You don't have access to the model with the specified model ID.`

**Cause**: Model access not granted in this account/region, or the IAM policy lacks `bedrock:InvokeModel` on the model ARN.

**Solution**: Enable the model under **Bedrock → Model access** in the correct region, and confirm the IAM policy lists the model (and inference-profile) ARN.

#### Issue 2: `ThrottlingException` / `TooManyRequestsException`

**Symptoms**: Intermittent failures under load, especially on popular frontier models.

**Cause**: On-demand per-model request/token quotas exceeded.

**Solution**: Use `retries={'mode': 'adaptive'}`, switch to a cross-region inference profile, request a quota increase, or buy Provisioned Throughput.

#### Issue 3: `ValidationException: provided model identifier is invalid`

**Symptoms**: A call that works in one region fails in another, or with a typo'd ID.

**Cause**: The model isn't offered in that region, or you used a bare `modelId` where an inference profile is required.

**Solution**: Verify with `list_foundation_models` / `list_inference_profiles` for the region and use the matching ID (often the `us.`-prefixed profile).

## Comparison with Alternatives

### How Amazon Bedrock Compares to Other Solutions

| Dimension | Amazon Bedrock | OpenAI / Anthropic APIs | Google Vertex AI | Self-host (SageMaker / EKS + vLLM) |
|-----------|----------------|-------------------------|------------------|-------------------------------------|
| Model choice | Many providers, one API | Single provider each | Gemini + Model Garden | Any open-weight model |
| Ops burden | Serverless, none | Serverless, none | Serverless, none | You run/scale GPUs |
| AWS integration | Native (IAM/VPC/CloudWatch) | External account/key | GCP-native | Native AWS |
| Data residency | In your AWS account | Provider's cloud | In your GCP project | Full control |
| Newest features | Slight lag behind providers | First to ship | First for Gemini | Depends on model |
| Cost model | Per-token / provisioned | Per-token | Per-token / provisioned | Pay for GPUs (fixed) |

### When to Choose This Tool

Choose Amazon Bedrock when:

- Your workload already lives on AWS and you want IAM/VPC/CloudWatch/billing integration out of the box.
- You want managed RAG, agents, and guardrails without building them.
- You value keeping model choice open behind one API and need data to stay inside your AWS account.

Prefer a **provider's own API** for day-one access to the newest features, or **self-hosting (SageMaker / EKS + vLLM)** when you need a model Bedrock doesn't offer or want full control over cost and latency at high volume.

## Resources

### Official Documentation

- Product page: https://aws.amazon.com/bedrock/
- Developer Guide: https://docs.aws.amazon.com/bedrock/latest/userguide/what-is-bedrock.html
- Converse API reference: https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html
- boto3 `bedrock-runtime` client: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-runtime.html

### Tutorials and Guides

- Knowledge Bases (managed RAG): https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html
- Agents for Amazon Bedrock: https://docs.aws.amazon.com/bedrock/latest/userguide/agents.html
- Guardrails: https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html
- Anthropic Claude on Bedrock: https://docs.anthropic.com/en/api/claude-on-amazon-bedrock

### Community Resources

- amazon-bedrock-samples (GitHub): https://github.com/aws-samples/amazon-bedrock-samples
- AWS Generative AI blog: https://aws.amazon.com/blogs/machine-learning/category/artificial-intelligence/generative-ai/
- AWS re:Post (Bedrock Q&A): https://repost.aws/

### Related Technologies

- Amazon SageMaker (train/host your own models) — see `sagemaker.ipynb`
- LangChain / LlamaIndex (Bedrock integrations for orchestration)
- Amazon OpenSearch Serverless / Aurora pgvector (Knowledge Base vector stores)